# CosyVoice2 Russian Audiobook Generator
Generates high-quality Russian audio with voice cloning using CosyVoice2-0.5B.

**Runtime: GPU (T4 or better)**

Each chapter WAV saves to Google Drive immediately.

In [ ]:
#@title 1. Mount Drive + Install CosyVoice2 + clone data repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
OUTPUT_DIR = '/content/drive/My Drive/tts_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted!')

# Clone CosyVoice with submodules
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd CosyVoice
!git submodule update --init --recursive

# System deps
!apt-get -qq -y install sox libsox-dev

# Remove heavy optional deps to save disk/time
!sed -i '/tensorrt/d' requirements.txt

# Install Python deps
!pip install -q -r requirements.txt

# Clone our data repo (chapter texts + ref audio)
!git clone https://github.com/stuk88/post-scarcity-architecture.git ../data_repo

print('Install complete!')

In [ ]:
#@title 2. Download CosyVoice2-0.5B model
from huggingface_hub import snapshot_download
snapshot_download('FunAudioLLM/CosyVoice2-0.5B',
                  local_dir='pretrained_models/CosyVoice2-0.5B')
print('Model downloaded!')

In [ ]:
#@title 3. Generate all chapters (saves each to Drive immediately)
import sys
sys.path.append('third_party/Matcha-TTS')

import gc, re, time, os
import torchaudio
import torch
from pathlib import Path
from cosyvoice.cli.cosyvoice import AutoModel

CHAPTER_DIR = Path('../data_repo/ru_tts_translation/chapter_texts')
REF_AUDIO = '../data_repo/ru_tts_translation/ref_audio.wav'
OUTPUT = Path('/content/drive/My Drive/tts_output')

# Transcript of the reference audio (from the abstract reading)
REF_TEXT = 'как структурное признание рынка и предлагает полноценную альтернативную архитектуру, протокольное финансирование через открытую лотерею. Некоммерческое машинное управление с ограниченными градиентами.'

print('Loading CosyVoice2-0.5B...')
cosyvoice = AutoModel(model_dir='pretrained_models/CosyVoice2-0.5B')
SR = cosyvoice.sample_rate
print(f'Model loaded! Sample rate: {SR}')

chapter_files = sorted(CHAPTER_DIR.glob('*.txt'))
print(f'\nGenerating {len(chapter_files)} chapters...\n')

t0 = time.time()

for ci, ch_file in enumerate(chapter_files):
    out_wav = OUTPUT / f'{ch_file.stem}.wav'
    if out_wav.exists():
        print(f'  [skip] {out_wav.name}')
        continue

    text = ch_file.read_text(encoding='utf-8').strip()
    segments = [s.strip() for s in text.split('\n\n') if s.strip()]
    print(f'  [{ci+1}/{len(chapter_files)}] {ch_file.name} ({len(text)} chars, {len(segments)} segs)')

    audio_chunks = []
    silence_short = torch.zeros(1, int(SR * 0.5))

    for si, seg in enumerate(segments):
        # Pause markers
        dots_only = seg.replace('.', '').replace(' ', '')
        if not dots_only:
            pause = min(seg.count('.') * 0.4, 3.0)
            audio_chunks.append(torch.zeros(1, int(SR * pause)))
            continue

        # Split long segments into chunks
        sentences = re.split(r'(?<=[.!?])\s+', seg)
        chunks, current = [], ''
        for s in sentences:
            if len(current) + len(s) > 300 and current:
                chunks.append(current.strip())
                current = s
            else:
                current = f'{current} {s}' if current else s
        if current.strip():
            chunks.append(current.strip())

        for chunk in chunks:
            try:
                for j in cosyvoice.inference_zero_shot(
                    chunk, REF_TEXT, REF_AUDIO, stream=False
                ):
                    audio_chunks.append(j['tts_speech'])
            except Exception as e:
                print(f'    ERR seg {si}: {str(e)[:80]}')

        audio_chunks.append(silence_short)

        if (si + 1) % 10 == 0:
            print(f'    {si+1}/{len(segments)} segs')

    # Save chapter to Drive immediately
    if audio_chunks:
        full = torch.cat(audio_chunks, dim=1)
        torchaudio.save(str(out_wav), full, SR)
        dur = full.shape[1] / SR
        elapsed = time.time() - t0
        print(f'    -> {out_wav.name} ({dur:.0f}s) [{elapsed:.0f}s elapsed]')
        del full, audio_chunks
        gc.collect()
        torch.cuda.empty_cache()

print(f'\nDone! All chapters in {time.time()-t0:.0f}s')
print(f'Files in Google Drive: tts_output/')

In [ ]:
#@title 4. Combine into MP3
import subprocess
from pathlib import Path

OUTPUT = Path('/content/drive/My Drive/tts_output')
wav_files = sorted(OUTPUT.glob('*.wav'))
print(f'Combining {len(wav_files)} chapters...')

with open('/tmp/filelist.txt', 'w') as f:
    for wav in wav_files:
        f.write(f"file '{wav}'\n")

mp3_path = OUTPUT / 'Post_Scarcity_v3_ru.mp3'
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', '/tmp/filelist.txt',
    '-codec:a', 'libmp3lame', '-qscale:a', '2',
    str(mp3_path)
], capture_output=True)

size = mp3_path.stat().st_size / (1024*1024)
print(f'Saved: {mp3_path} ({size:.1f} MB)')